In [ ]:
import requests
import os
import pandas as pd
from PIL import Image
import io
import numpy as np
from tqdm import tqdm

In [ ]:
urls = {
    "fold1": "https://huggingface.co/datasets/RationAI/PanNuke/resolve/refs%2Fconvert%2Fparquet/default/fold1/0000.parquet?download=true",
    "fold2": "https://huggingface.co/datasets/RationAI/PanNuke/resolve/refs%2Fconvert%2Fparquet/default/fold2/0000.parquet?download=true",
    "fold3": "https://huggingface.co/datasets/RationAI/PanNuke/resolve/refs%2Fconvert%2Fparquet/default/fold3/0000.parquet?download=true"
}

for fold, url in urls.items():
    response = requests.get(url, stream=True)
    filename = f"{fold}.parquet"

    with open(filename, "wb") as file:
        for chunk in response.iter_content(chunk_size=8192):
            file.write(chunk)
    
    print(f"Downloaded {filename}")

Downloaded fold1.parquet
Downloaded fold2.parquet
Downloaded fold3.parquet


In [ ]:
# Define dataset files
parquet_files = ["fold1.parquet", "fold2.parquet", "fold3.parquet"]

# Define output directories
output_dir = "dataset"
os.makedirs(output_dir, exist_ok=True)

# Create a list to store metadata for the CSV
data_records = []

# Process each Parquet file
for parquet_file in tqdm(parquet_files):
    df = pd.read_parquet(parquet_file)
    
    for index, row in df.iterrows():
        # Create unique filename
        image_id = f"{parquet_file.replace('.parquet', '')}_{index}"
        image_folder = os.path.join(output_dir, image_id)
        os.makedirs(image_folder, exist_ok=True)

        # Save the main image
        image_bytes = row["image"]["bytes"]
        image = Image.open(io.BytesIO(image_bytes))
        image_path = os.path.join(image_folder, "image.png")
        image.save(image_path)

        # Create masks directory
        mask_dir = os.path.join(image_folder, "masks")
        os.makedirs(mask_dir, exist_ok=True)

        # Save masks
        for mask_index, mask_bytes in enumerate(row["instances"]):
            mask = Image.open(io.BytesIO(mask_bytes["bytes"])).convert("L")  # Convert to grayscale
            mask_category = row["categories"][mask_index]  # Get category
            mask_path = os.path.join(mask_dir, f"mask_{mask_category}_{mask_index}.png")
            mask.save(mask_path)

        # Append metadata to CSV records
        data_records.append([image_path, row["categories"], row["tissue"]])

# Save CSV file
csv_path = os.path.join(output_dir, "metadata.csv")
df_metadata = pd.DataFrame(data_records, columns=["file_path", "category", "tissue"])
df_metadata.to_csv(csv_path, index=False)

print(f"Dataset prepared! Images and masks are saved in '{output_dir}', and metadata is in '{csv_path}'.")

100%|██████████| 3/3 [06:50<00:00, 136.69s/it]


Dataset prepared! Images and masks are saved in 'dataset', and metadata is in 'dataset/metadata.csv'.


In [3]:
!rm -r ../working/*.parquet